# 07.3 — Load parameters

The first curriculum lever ([07.1](07.1_curriculum_learning_intuition.ipynb)) is **LoadParameters** — a schedule of *data-augmentation* strengths. Each epoch, the training data is corrupted with a controlled amount of synthetic noise (channel offsets, white noise, drift), and the *amount* is scheduled: the Soft Three-Stage regime goes **clean → noisy → clean**. Two things make this lever subtle: the augmentation is read **live inside `__getitem__`** (Critical Note #8), so a per-epoch schedule change hits the very next batch with no DataLoader rebuild; and only the *training* data is augmented — validation and test stay clean.

This notebook covers:

* What data augmentation is and why few-trial neural data needs it.
* The augmentation types: channel offset, white noise, random walk, time shift.
* The **live-read contract** (Critical Note #8) — augmentation read per-batch.
* The clean → noisy → clean curriculum, and why train-only.

**Prerequisites:** [07.2 piecewise schedules](07.2_piecewise_linear_schedules.ipynb), [03.1 the dataset](../03_data_pipeline/03.1_dataset_vs_filedatastore.ipynb).

## Section 1 — What MATLAB does

`cgg_generateLoadParameters_v2` builds a set of `CurrentSTD*` values that vary per epoch; `cgg_generateDataAugmentationSignal` uses them to synthesize an additive noise tensor at load time:

```matlab
% Per-epoch augmentation strengths (a schedule, not constants):
LoadParameters = cgg_generateLoadParameters_v2(...);
STD_white = cgg_calculateDynamicValue(LoadParameters.WhiteNoise, epoch);

% At data-load time, build and ADD a noise signal:
aug = cgg_generateDataAugmentationSignal(size(X), STD_offset, STD_white, STD_walk);
X_augmented = X + aug;    % training data only
```

The port maps `CurrentSTD*` → `make_load_schedule(std_channel_offset=, std_white_noise=, std_random_walk=, std_time_shift=)` and `cgg_generateDataAugmentationSignal` → `additive_augmentation_signal`. `NaN` disables an augmentation (the MATLAB default).

## Section 2 — The Python concepts you need

### 2.1 — Why augment neural data

Neural decoding datasets are *small* — a session might have a few hundred trials, far fewer than the millions of images a vision model trains on. With so little data, a model easily memorizes the training trials instead of learning the signal. **Augmentation** manufactures variety: it adds realistic noise to each trial so the model sees a slightly different version every epoch and is forced to learn what's *invariant* (the decodable signal) rather than the exact noise pattern of each recording. It's regularization, like the VAE's sampling ([06.3](../06_loss_orchestration/06.3_stochastic_vs_deterministic_placement.ipynb)) and weight decay ([06.8](../06_loss_orchestration/06.8_l2_inside_the_loss_kernel.ipynb)), but applied to the *inputs*.

### 2.2 — The augmentation types

Each simulates a real neural-recording artifact:

| Augmentation | What it adds | Simulates |
|---|---|---|
| **channel offset** | a per-channel constant (broadcast over time) | DC bias / electrode baseline differences |
| **white noise** | independent per-sample Gaussian | measurement / thermal noise |
| **random walk** | cumulative-sum of Gaussian steps (drift) | slow baseline wander |
| **time shift** | jitter in trial alignment | trial-to-trial timing variability |

The first three are summed and low-pass smoothed by `additive_augmentation_signal`; time shift is applied separately at load time. Each is toggled by its `std_*` (NaN = off) and scaled by its scheduled magnitude.

In [ ]:
import numpy as np
from neural_data_decoding.data.augmentation import additive_augmentation_signal

rng = np.random.default_rng(0)
shape = (1, 200, 1)   # (channels, samples, probes) — one channel, 200 timesteps

offset = additive_augmentation_signal(shape, std_channel_offset=1.0, std_white_noise=None, std_random_walk=None, rng=np.random.default_rng(1), want_low_pass=False)
white  = additive_augmentation_signal(shape, std_channel_offset=None, std_white_noise=1.0, std_random_walk=None, rng=np.random.default_rng(2), want_low_pass=False)
walk   = additive_augmentation_signal(shape, std_channel_offset=None, std_white_noise=None, std_random_walk=1.0, rng=np.random.default_rng(3), want_low_pass=False)
print("channel offset — constant over time? std across samples =", round(offset.std(), 4), "(≈0: it's a constant)")
print("white noise    — per-sample, std =", round(white.std(), 3))
print("random walk    — drifts, range =", round(walk.ptp(), 2), "(cumulative sum wanders)")

### 2.3 — Draw the three augmentations

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharey=False)
for ax, sig, title, color in [
    (axes[0], offset[0, :, 0], "channel offset (constant)", "#3366aa"),
    (axes[1], white[0, :, 0], "white noise (per-sample)", "#aa3366"),
    (axes[2], walk[0, :, 0], "random walk (drift)", "#33aa66")]:
    ax.plot(sig, color=color, lw=1)
    ax.axhline(0, color="gray", ls=":", lw=0.8)
    ax.set_title(title); ax.set_xlabel("timestep (sample)")
axes[0].set_ylabel("added value")
plt.tight_layout(); plt.show()
print("Three qualitatively different corruptions: a flat shift, jagged noise, and a slow wander.")

### 2.4 — The live-read contract (Critical Note #8)

Here's the subtle design point. You might expect augmentation strength to be fixed when the `Dataset` is built. It isn't. The `Dataset` reads `schedule.current("std_*")` **inside every `__getitem__` call** — so when the training loop calls `schedule.update(epoch)` at the top of an epoch, the *next batch* immediately reflects the new strength. No `DataLoader` rebuild, no dataset re-instantiation. The schedule and the data pipeline are coupled *live*.

In [ ]:
from neural_data_decoding.training.schedules.factory import make_load_schedule, ScheduleWaypoints

# clean → noisy → clean: waypoints [5,10,25,45], magnitudes [0.01, 1.0, 1.0, 0.01]
wp = ScheduleWaypoints.of((5, 10, 25, 45), (0.01, 1.0, 1.0, 0.01))
sched = make_load_schedule(std_white_noise=1.0, waypoints=wp)

# Simulate what the Dataset does: read sched.current() at "getitem time".
def getitem_augmentation_strength():
    return sched.current("std_white_noise")     # read LIVE, every call

for epoch in (1, 10, 20, 45):
    sched.update(epoch)                          # loop calls this once per epoch
    print(f"epoch {epoch:2}: next batch's white-noise std = {getitem_augmentation_strength():.3f}")
print("→ update() at epoch start → the very next __getitem__ sees the new strength (no rebuild).")

That's `clean → noisy → clean`: barely any noise early (epoch 1: `0.01`), full strength in the middle (epoch 20: `1.0`), tapering back toward clean by the end (epoch 45: `0.059`, reaching the `0.01` floor at epoch 46 — the [07.2](07.2_piecewise_linear_schedules.ipynb) off-by-one lag showing up in a real schedule). The curriculum rationale: learn the clean signal first, then build robustness against heavy noise, then fine-tune on near-clean data that matches the real test distribution. The augmentation *magnitude* schedule is just [07.2](07.2_piecewise_linear_schedules.ipynb)'s interpolator applied to the load lever.

### 2.5 — Train-only augmentation

Only the *training* dataset gets a load schedule. Validation and test data are **never** augmented — you evaluate on the real, clean distribution, not a corrupted one. Augmenting the validation set would make the metric measure "accuracy on noise," not true decoding performance, and would make runs non-comparable. This mirrors the train/eval split of every other stochastic mechanism ([06.13](../06_loss_orchestration/06.13_sampling_layer_deterministic_at_inference.ipynb)): randomness in training, determinism in evaluation.

## Section 3 — The `neural_data_decoding` implementation

`make_load_schedule` — the four `std_*` bases, NaN-disabled by default:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/training/schedules/factory.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if "def make_load_schedule" in line)
for k in range(i, i + 7):
    print(f"{k + 1:4} | {src[k]}")
print("...")
# the Dataset's live read:
ds = Path("../../src/neural_data_decoding/data/dataset.py").read_text().splitlines()
j0 = next(j for j, line in enumerate(ds) if "def _current_or_none" in line)
for k in range(j0, j0 + 3):
    print(f"{k + 1:4} | {ds[k]}")

Things to spot:

* `make_load_schedule(*, std_channel_offset=nan, std_white_noise=nan, std_random_walk=nan, std_time_shift=nan, waypoints=None)` — every augmentation defaults to `NaN` (off); you opt in by passing a base and a schedule.
* `_current_or_none` in the dataset reads `sched.current(name)` **inside `__getitem__`** — the live-read (§2.4, Critical Note #8). A missing key contributes `0` (NaN-disable parity).
* `additive_augmentation_signal` (in `data/augmentation.py`, port of `cgg_generateDataAugmentationSignal.m`) sums the enabled components and Gaussian-smooths them — the channel offset broadcasts over samples, white noise is per-sample, the random walk is a `cumsum`.
* The schedule uses a *shared* `ScheduleWaypoints` in the Soft Three-Stage regime (all augmentations ramp together); other regimes can give each `std_*` its own waypoints ([07.2 §2.2](07.2_piecewise_linear_schedules.ipynb)'s `WaypointConfig` shapes).
* Only the training `Dataset` is constructed with a `load_schedule`; the val/test datasets get `None` (§2.5).

## Section 4 — Hands-on exercises

### Exercise 4.1 — noise strength tracks the schedule

Build a `std_white_noise` schedule and confirm the augmentation signal's actual standard deviation follows the scheduled magnitude across epochs.

In [ ]:
# Reveal:
sched = make_load_schedule(std_white_noise=1.0, waypoints=ScheduleWaypoints.of((1, 10), (0.0, 1.0)))
for epoch in (1, 5, 10):
    sched.update(epoch)
    std = sched.current("std_white_noise")
    sig = additive_augmentation_signal((4, 500, 2), std_channel_offset=None,
        std_white_noise=std, std_random_walk=None, rng=np.random.default_rng(0), want_low_pass=False)
    print(f"epoch {epoch:2}: scheduled std={std:.2f} → actual signal std={sig.std():.3f}")
print("→ the augmentation magnitude faithfully tracks the schedule.")

### Exercise 4.2 — NaN disables an augmentation

Show that passing `None` (the NaN-disable) for an augmentation contributes exactly zero to the signal.

In [ ]:
# Reveal:
only_white = additive_augmentation_signal((2, 100, 1), std_channel_offset=None,
    std_white_noise=1.0, std_random_walk=None, rng=np.random.default_rng(0), want_low_pass=False)
all_off = additive_augmentation_signal((2, 100, 1), std_channel_offset=None,
    std_white_noise=None, std_random_walk=None, rng=np.random.default_rng(0), want_low_pass=False)
print("white-only signal std:", round(only_white.std(), 3))
print("all-disabled signal — all zero?", bool((all_off == 0).all()), "(NaN/None = off)")

## Section 5 — Common errors

### Changing the schedule mid-training has no effect

The live-read requires the `Dataset` to actually read `schedule.current()` inside `__getitem__` (§2.4). If augmentation strength was baked in at construction, `schedule.update()` won't reach the data. Confirm the dataset holds the *schedule*, not a snapshotted std.

### Validation accuracy looks suspiciously low / noisy

The validation set may be getting augmented (§2.5). Only the training dataset should carry a load schedule; val/test must be clean.

### Augmentation too strong early → the model never learns

A curriculum that starts at full noise (instead of clean) drowns the signal before the model finds it. Start clean and ramp up ([07.1 §2.2](07.1_curriculum_learning_intuition.ipynb)) — the Soft Three-Stage `[0.01, 1.0, 1.0, 0.01]` shape.

### Reused RNG gives correlated augmentations

Each `__getitem__` should draw fresh noise. Sharing a poorly-managed generator (or re-seeding identically every call) makes every trial get the *same* noise — no augmentation benefit. The dataset threads a long-lived generator.

### Forgetting a `std_*` key that the schedule references

A schedule with waypoints for `std_time_shift` but no `std_time_shift` base contributes 0 for it. Not an error (NaN-disable), but if you *meant* to augment, the base is missing.

## Section 6 — Further reading

- [`src/neural_data_decoding/data/augmentation.py`](../../src/neural_data_decoding/data/augmentation.py) — `additive_augmentation_signal` (Critical Note #7).
- [`src/neural_data_decoding/training/schedules/factory.py`](../../src/neural_data_decoding/training/schedules/factory.py) — `make_load_schedule`.
- [03.1 the dataset protocol](../03_data_pipeline/03.1_dataset_vs_filedatastore.ipynb) — where the live-read `__getitem__` lives.

Next up: **[07.4 loss weights curriculum](07.4_loss_weights_curriculum.ipynb)** — the second lever: scheduling the per-component loss weights, including KL annealing.